# OLAY BAZLI RANDOM FOREST: SADECE BÜYÜKLÜK TAHMİNİ

Burada hiçbir gruplama ve ortalama yok. Her satır **tek bir deprem**; hedef o depremin büyüklüğü, özellikler ise ondan önceki son 5 depremin ham değerleri (büyüklük, derinlik, enlem, boylam ve aradan geçen süre).

İki soru deneniyor:
- **Küresel sıra:** dünyada kaydedilen bir sonraki depremin büyüklüğü (önceki 5 deprem nerede olursa olsun).
- **Bölgesel sıra:** aynı 1°×1° hücrede kaydedilen bir sonraki depremin büyüklüğü (o hücredeki önceki 5 deprem).

Bölme kronolojik: ilk %80 deprem train, son %20 test. Hızlı çalışması için grid küçük (4 kombinasyon, 100 ağaç).

In [1]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from regional import regression_metrics, classification_metrics

N_PREV = 5   # kaç önceki depreme bakılacağı

C:\Users\eren1\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
def make_event_rows(df, per_cell):
    # Her deprem için önceki N_PREV depremin ham değerleri (ortalama yok)
    df = df.dropna(subset=['latitude', 'longitude', 'depth', 'mag', 'time']).sort_values('time').reset_index(drop=True)
    hours = (df['time'] - df['time'].min()).dt.total_seconds() / 3600
    data = pd.DataFrame({'target_mag': df['mag'], 'time': df['time']})
    if per_cell:
        cell = [np.floor(df['latitude']), np.floor(df['longitude'])]
        shift = lambda s, k: s.groupby(cell).shift(k)
        # hangi bölgeye bakıldığı biliniyor (hücrenin köşe koordinatı)
        data['cell_lat'], data['cell_lon'] = cell
    else:
        shift = lambda s, k: s.shift(k)
    for k in range(1, N_PREV + 1):
        data[f'mag_{k}'] = shift(df['mag'], k)
        data[f'depth_{k}'] = shift(df['depth'], k)
        data[f'lat_{k}'] = shift(df['latitude'], k)
        data[f'lon_{k}'] = shift(df['longitude'], k)
        # k. önceki deprem ile bir sonrakinin arası (saat); en yenisi için tahmin anı bilinmediğinden kullanılmıyor
        if k > 1:
            data[f'gap_{k}'] = shift(hours, k - 1) - shift(hours, k)
    return data.dropna().reset_index(drop=True)


def run_event_rf(df, name, per_cell, model_path):
    data = make_event_rows(df, per_cell)
    cut = int(0.8 * len(data))            # satırlar zamana göre sıralı: ilk %80 train, son %20 test
    train, test = data[:cut], data[cut:]
    features = [c for c in data.columns if c not in ('target_mag', 'time')]

    search = GridSearchCV(
        RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1),
        {'min_samples_leaf': [5, 25], 'max_features': [0.4, 0.8]},
        cv=TimeSeriesSplit(n_splits=3), scoring='neg_mean_squared_error', error_score='raise', n_jobs=-1)
    search.fit(train[features], train['target_mag'])

    thresholds = [train['target_mag'].quantile(0.75), train['target_mag'].quantile(0.90)]
    rows_reg, rows_cls = [], []
    predictions = {'Random Forest': (search.predict(train[features]), search.predict(test[features])),
                   'Önceki deprem (referans)': (train['mag_1'].values, test['mag_1'].values),
                   'Train ortalaması (referans)': (np.full(len(train), train['target_mag'].mean()),
                                                    np.full(len(test), train['target_mag'].mean()))}
    for model, preds in predictions.items():
        for split, part, pred in (('Train', train, preds[0]), ('Test', test, preds[1])):
            rows_reg.append({'Model': model, 'Veri': split, **regression_metrics(part['target_mag'], pred)})
            for t in thresholds:
                rows_cls.append({'Model': model, 'Veri': split, 'Eşik': f'≥{t:.2f}',
                                 **classification_metrics(part['target_mag'], pred, t)})
    print(f"\n=== {name} | train: {len(train)} deprem, test: {len(test)} deprem, {len(features)} özellik ===")
    print(f"En iyi parametreler: {search.best_params_}")
    print(pd.DataFrame(rows_reg).set_index(['Model', 'Veri']).round(3).to_string())
    print()
    print(pd.DataFrame(rows_cls).set_index(['Model', 'Veri', 'Eşik']).round(3).to_string())
    importance = pd.Series(search.best_estimator_.feature_importances_, index=features)
    print("\nEn önemli 5 özellik:", importance.sort_values(ascending=False).head(5).round(3).to_dict())
    joblib.dump(search.best_estimator_, model_path, compress=3)
    return search

## DATASET 1 (USGS, 2022)

In [3]:
df1 = pd.read_csv('usgs_main.csv')
df1['time'] = pd.to_datetime(df1['time'])
df1 = df1[df1['type'] == 'earthquake']   # taş ocağı patlaması vb. kayıtlar deprem değil
rf_1_global = run_event_rf(df1, 'Olay RF Dataset1 - küresel sıra', per_cell=False, model_path='models/event_rf_dataset1_global.pkl')
rf_1_cell = run_event_rf(df1, 'Olay RF Dataset1 - bölgesel sıra', per_cell=True, model_path='models/event_rf_dataset1_cell.pkl')


=== Olay RF Dataset1 - küresel sıra | train: 59786 deprem, test: 14947 deprem, 24 özellik ===
En iyi parametreler: {'max_features': 0.4, 'min_samples_leaf': 25}
                                      R²    MAE    MSE   RMSE
Model                       Veri                             
Random Forest               Train  0.208  0.788  1.155  1.075
                            Test   0.001  0.880  1.480  1.216
Önceki deprem (referans)    Train -0.874  1.211  2.734  1.653
                            Test  -0.878  1.229  2.780  1.667
Train ortalaması (referans) Train  0.000  0.892  1.459  1.208
                            Test  -0.012  0.884  1.498  1.224

                                         Pozitif oranı  Accuracy  Precision  Recall     F1    AUC
Model                       Veri  Eşik                                                           
Random Forest               Train ≥2.08          0.251     0.760      0.973   0.045  0.087  0.886
                                  ≥3.20        


=== Olay RF Dataset1 - bölgesel sıra | train: 54466 deprem, test: 13617 deprem, 26 özellik ===
En iyi parametreler: {'max_features': 0.4, 'min_samples_leaf': 25}
                                      R²    MAE    MSE   RMSE
Model                       Veri                             
Random Forest               Train  0.743  0.348  0.239  0.489
                            Test   0.733  0.382  0.289  0.538
Önceki deprem (referans)    Train  0.465  0.505  0.496  0.704
                            Test   0.571  0.490  0.466  0.682
Train ortalaması (referans) Train  0.000  0.722  0.928  0.963
                            Test  -0.030  0.769  1.118  1.057

                                         Pozitif oranı  Accuracy  Precision  Recall     F1    AUC
Model                       Veri  Eşik                                                           
Random Forest               Train ≥1.90          0.260     0.874      0.813   0.670  0.735  0.912
                                  ≥2.48       

## DATASET 2 (M≥5.5, 1900-2023)

In [4]:
df2 = pd.read_csv('Significant Earthquake Dataset 1900-2023.csv')
df2 = df2.rename(columns={'Time':'time','Mag':'mag','Depth':'depth','Latitude':'latitude','Longitude':'longitude'})
df2['time'] = pd.to_datetime(df2['time'])
rf_2_global = run_event_rf(df2, 'Olay RF Dataset2 - küresel sıra', per_cell=False, model_path='models/event_rf_dataset2_global.pkl')
rf_2_cell = run_event_rf(df2, 'Olay RF Dataset2 - bölgesel sıra', per_cell=True, model_path='models/event_rf_dataset2_cell.pkl')


=== Olay RF Dataset2 - küresel sıra | train: 29753 deprem, test: 7439 deprem, 24 özellik ===
En iyi parametreler: {'max_features': 0.4, 'min_samples_leaf': 25}
                                      R²    MAE    MSE   RMSE
Model                       Veri                             
Random Forest               Train  0.276  0.289  0.153  0.391
                            Test  -0.016  0.321  0.185  0.431
Önceki deprem (referans)    Train -0.706  0.428  0.360  0.600
                            Test  -0.996  0.420  0.364  0.603
Train ortalaması (referans) Train  0.000  0.349  0.211  0.459
                            Test  -0.044  0.344  0.190  0.436

                                         Pozitif oranı  Accuracy  Precision  Recall     F1    AUC
Model                       Veri  Eşik                                                           
Random Forest               Train ≥6.20          0.252     0.782      0.711   0.228  0.345  0.825
                                  ≥6.60         


=== Olay RF Dataset2 - bölgesel sıra | train: 17440 deprem, test: 4360 deprem, 26 özellik ===
En iyi parametreler: {'max_features': 0.4, 'min_samples_leaf': 25}
                                      R²    MAE    MSE   RMSE
Model                       Veri                             
Random Forest               Train  0.212  0.277  0.142  0.377
                            Test   0.007  0.308  0.170  0.413
Önceki deprem (referans)    Train -0.816  0.401  0.327  0.572
                            Test  -0.928  0.406  0.331  0.575
Train ortalaması (referans) Train  0.000  0.317  0.180  0.424
                            Test  -0.006  0.314  0.172  0.415

                                         Pozitif oranı  Accuracy  Precision  Recall     F1    AUC
Model                       Veri  Eşik                                                           
Random Forest               Train ≥6.10          0.261     0.760      0.736   0.127  0.217  0.813
                                  ≥6.47        